# Tarea 2 — IA

## Pregunta 1: Regresión lineal y regresión logística

**Integrantes:** Fernando Ortega, Ihan San Martín

**Dataset:** Hotel Booking Demand (119.390 reservas, 32 columnas). Fuente: Antonio, Almeida & Nunes (2019), *Data in Brief*; copia pública en TidyTuesday (GitHub).

### Declaración de uso de IA generativa
Se utilizó IA generativa solo para instalar el entorno (Python, Jupyter y librerías) buscar fuentes de datos públicas y verificar que el dataset cumple el tamaño mínimo, el diseño y la solucion fue hecha por nosotros. 

## 0. Librerías

In [1]:
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LinearRegression, LogisticRegression
from sklearn.metrics import r2_score, mean_squared_error, mean_absolute_error
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_auc_score, confusion_matrix

semilla = 42

## 1. Carga de datos
Cargamos el csv directo desde internet, así el notebook corre en cualquier computador sin tener que bajar nada a mano.

In [2]:
url = "https://raw.githubusercontent.com/rfordatascience/tidytuesday/master/data/2020/2020-02-11/hotels.csv"
df = pd.read_csv(url)
print(df.shape)
df.head()

(119390, 32)


,hotel,is_canceled,lead_time,arrival_date_year,arrival_date_month,arrival_date_week_number,arrival_date_day_of_month,stays_in_weekend_nights,stays_in_week_nights,adults,...,deposit_type,agent,company,days_in_waiting_list,customer_type,adr,required_car_parking_spaces,total_of_special_requests,reservation_status,reservation_status_date
0,Resort Hotel,0,342,2015,July,27,1,0,0,2,...,No Deposit,NaN,NaN,0,Transient,0.0,0,0,Check-Out,2015-07-01
1,Resort Hotel,0,737,2015,July,27,1,0,0,2,...,No Deposit,NaN,NaN,0,Transient,0.0,0,0,Check-Out,2015-07-01
2,Resort Hotel,0,7,2015,July,27,1,0,1,1,...,No Deposit,NaN,NaN,0,Transient,75.0,0,0,Check-Out,2015-07-02
3,Resort Hotel,0,13,2015,July,27,1,0,1,1,...,No Deposit,304.0,NaN,0,Transient,75.0,0,0,Check-Out,2015-07-02
4,Resort Hotel,0,14,2015,July,27,1,0,2,2,...,No Deposit,240.0,NaN,0,Transient,98.0,0,1,Check-Out,2015-07-03


In [3]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 119390 entries, 0 to 119389
Data columns (total 32 columns):
 #   Column                          Non-Null Count   Dtype  
---  ------                          --------------   -----  
 0   hotel                           119390 non-null  str    
 1   is_canceled                     119390 non-null  int64  
 2   lead_time                       119390 non-null  int64  
 3   arrival_date_year               119390 non-null  int64  
 4   arrival_date_month              119390 non-null  str    
 5   arrival_date_week_number        119390 non-null  int64  
 6   arrival_date_day_of_month       119390 non-null  int64  
 7   stays_in_weekend_nights         119390 non-null  int64  
 8   stays_in_week_nights            119390 non-null  int64  
 9   adults                          119390 non-null  int64  
 10  children                        119386 non-null  float64
 11  babies                          119390 non-null  int64  
 12  meal                       

In [4]:
nulos = df.isna().sum()
print(nulos[nulos > 0])

children         4
country        488
agent        16340
company     112593
dtype: int64


Lo primero que vimos es que hay 4 columnas con nulos. company está casi vacía y agent también tiene harto, y además las dos son códigos de la agencia o la empresa, así que no son números que signifiquen algo como una cantidad. children tiene solo 4 nulos y country unos cientos.

In [5]:
print(df.nunique().sort_values())

hotel                                2
is_canceled                          2
is_repeated_guest                    2
arrival_date_year                    3
deposit_type                         3
reservation_status                   3
customer_type                        4
babies                               5
meal                                 5
distribution_channel                 5
children                             5
required_car_parking_spaces          5
total_of_special_requests            6
market_segment                       8
reserved_room_type                  10
arrival_date_month                  12
assigned_room_type                  12
adults                              14
previous_cancellations              15
stays_in_weekend_nights             17
booking_changes                     21
arrival_date_day_of_month           31
stays_in_week_nights                35
arrival_date_week_number            53
previous_bookings_not_canceled      73
days_in_waiting_list     

Con esto nos hicimos una idea de qué columnas son continuas, que son las que tienen muchos valores distintos como adr o lead_time, y cuáles son binarias o de categorías como hotel, is_canceled o meal.

## 2. Limpieza y preprocesamiento
Antes de separar los datos hicimos una limpieza fila por fila, o sea cosas que no dependen de sacar promedios ni nada del total, así no estamos mirando los datos de prueba sin querer.

In [6]:
print("filas duplicadas:", df.duplicated().sum())

personas = df["adults"] + df["children"].fillna(0) + df["babies"]
print("reservas sin personas:", (personas == 0).sum())

print(df["adr"].describe())
print("adr negativo:", (df["adr"] < 0).sum())
print("adr igual a 0:", (df["adr"] == 0).sum())
print("adr mayor a 1000:", (df["adr"] > 1000).sum())

filas duplicadas: 31994
reservas sin personas: 180
count    119390.000000
mean        101.831122
std          50.535790
min          -6.380000
25%          69.290000
50%          94.575000
75%         126.000000
max        5400.000000
Name: adr, dtype: float64
adr negativo: 1
adr igual a 0: 1959
adr mayor a 1000: 1


Lo que encontramos aquí fue que hay casi 32 mil filas repetidas exactamente iguales. Como no hay un ID de reserva no hay forma de saber si son reservas distintas o copias, y si las dejamos el modelo podría ver la misma fila en train y en validación y los resultados saldrían mejores de lo que son, así que las sacamos. También hay 180 reservas con 0 personas, que no tiene sentido, así que también las sacamos. Y el adr tiene un valor negativo, casi 2 mil en 0 y uno de 5400 que claramente es un error, entonces nos quedamos solo con los adr entre 0 y 1000 (sin contar el 0).

In [7]:
datos = df.drop_duplicates()
datos["children"] = datos["children"].fillna(0)
personas = datos["adults"] + datos["children"] + datos["babies"]
datos = datos[personas > 0]
datos = datos[(datos["adr"] > 0) & (datos["adr"] < 1000)]

print("antes:", df.shape)
print("despues:", datos.shape)

antes: (119390, 32)
despues: (85585, 32)


Después sacamos algunas columnas. reservation_status y reservation_status_date dicen si la reserva terminó en Canceled, Check-Out o No-Show, que sería como tener la respuesta de si se canceló o no, y eso solo se sabe después, así que dejarlas sería hacer trampa. company y agent son casi puros nulos y son códigos, y country tiene más de 170 países distintos, así que lo dejamos fuera para no complicarnos con tantas categorías.

In [8]:
datos = datos.drop(columns=["reservation_status", "reservation_status_date", "company", "agent", "country"])

datos["hotel_city"] = (datos["hotel"] == "City Hotel").astype(int)
datos["deposito_no_reemb"] = (datos["deposit_type"] == "Non Refund").astype(int)
datos["temporada_alta"] = datos["arrival_date_month"].isin(["June", "July", "August"]).astype(int)

print(datos.shape)

(85585, 30)


Lo que hicimos aquí fue crear 3 columnas nuevas a partir de columnas de texto, porque los modelos lineales necesitan números. hotel_city es 1 si es el hotel de ciudad y 0 si es el resort, deposito_no_reemb es 1 si la persona pagó un depósito que no le devuelven, y temporada_alta es 1 si llega en junio, julio o agosto.

El tema de temporada_alta es que el mes solo sería un número del 1 al 12 que no sube ni baja en línea recta con el precio, entonces lo pasamos a algo de sí o no.

## 3. División 70% entrenamiento / 15% validación / 15% prueba
Primero separamos un 70% para entrenar y el 30% que sobra lo partimos a la mitad, 15% para validación y 15% para prueba. Desde aquí en adelante todo lo que decidimos lo hacemos mirando solo train.

In [9]:
train, resto = train_test_split(datos, test_size=0.30, random_state=semilla)
val, test = train_test_split(resto, test_size=0.50, random_state=semilla)

print("train:", len(train))
print("val:", len(val))
print("test:", len(test))

train: 59909
val: 12838
test: 12838


## 4. Variables objetivo y predictoras (mirando solo entrenamiento)
### 4.1 Y1 continua: adr

In [10]:
print(train["adr"].describe())
print(train.groupby("arrival_date_month")["adr"].mean().round(1))

count    59909.000000
mean       108.543060
std         50.255178
min          0.260000
25%         74.400000
50%         99.070000
75%        135.000000
max        510.000000
Name: adr, dtype: float64
arrival_date_month
April        105.1
August       153.0
December      84.7
February      76.5
January       72.1
July         138.0
June         121.6
March         82.1
May          113.1
November      74.9
October       93.3
September    114.7
Name: adr, dtype: float64


Elegimos Y1 = adr, que es lo que se paga en promedio por noche. Es la continua que más sentido tiene predecir porque tiene muchísimos valores distintos, va más o menos de 0 a 500 y a un hotel le sirve saber cuánto va a pagar alguien según su reserva.

En los números se ve que la mitad de las reservas está entre 74 y 135 euros, y que en julio y agosto el precio promedio sube harto (sobre 135) comparado con invierno, que anda por los 75. Eso nos confirmó que la idea de temporada_alta iba bien.

### 4.2 Y2 binaria: is_canceled

In [11]:
print(train["is_canceled"].value_counts())
print(train["is_canceled"].value_counts(normalize=True).round(3))

is_canceled
0    43202
1    16707
Name: count, dtype: int64
is_canceled
0    0.721
1    0.279
Name: proportion, dtype: float64


Teníamos 3 opciones binarias, hotel, is_canceled e is_repeated_guest. Elegimos Y2 = is_canceled (1 si canceló y 0 si no) porque es la más interesante, a un hotel le sirve saber qué reservas se le van a caer. hotel no tiene mucho sentido predecirlo porque ya se sabe en qué hotel reservaste, e is_repeated_guest está muy desbalanceada porque casi todos son 0.

En train cerca del 28% de las reservas se cancela, o sea que no está 50 y 50, y eso hay que tenerlo en cuenta porque un modelo que diga que nadie cancela ya le achuntaría al 72%.

### 4.3 Elección de X1, ..., X10

In [12]:
correlaciones = train.corr(numeric_only=True)
print(correlaciones["adr"].sort_values().round(3))

is_repeated_guest                -0.127
previous_bookings_not_canceled   -0.080
previous_cancellations           -0.048
deposito_no_reemb                -0.045
days_in_waiting_list             -0.032
lead_time                         0.008
stays_in_weekend_nights           0.018
booking_changes                   0.021
arrival_date_day_of_month         0.021
babies                            0.029
stays_in_week_nights              0.031
required_car_parking_spaces       0.039
arrival_date_week_number          0.121
hotel_city                        0.122
is_canceled                       0.124
total_of_special_requests         0.150
arrival_date_year                 0.181
adults                            0.328
children                          0.362
temporada_alta                    0.436
adr                               1.000
Name: adr, dtype: float64


In [13]:
print(correlaciones["is_canceled"].sort_values().round(3))

required_car_parking_spaces      -0.187
total_of_special_requests        -0.122
booking_changes                  -0.095
is_repeated_guest                -0.081
previous_bookings_not_canceled   -0.050
babies                           -0.022
arrival_date_week_number          0.002
days_in_waiting_list              0.005
arrival_date_day_of_month         0.006
previous_cancellations            0.052
stays_in_weekend_nights           0.060
children                          0.063
temporada_alta                    0.064
hotel_city                        0.075
stays_in_week_nights              0.081
adults                            0.083
arrival_date_year                 0.088
adr                               0.124
deposito_no_reemb                 0.164
lead_time                         0.180
is_canceled                       1.000
Name: is_canceled, dtype: float64


Lo que hicimos aquí fue ver la correlación de cada columna numérica con las dos Y usando solo train. Como hay que usar las mismas 10 variables para los dos modelos, buscamos un grupo que tuviera algunas buenas para el precio y otras buenas para la cancelación.

Para el precio elegimos adults y children, porque más gente es una habitación más grande y más cara, y son las que más se correlacionan con adr. También temporada_alta porque en verano sube harto el precio, hotel_city porque el de ciudad es más caro en promedio y además cancela más, e is_repeated_guest porque los que repiten pagan menos y cancelan menos.

Para la cancelación elegimos lead_time, que son los días entre que reservas y llegas y es de las que más se relaciona con cancelar (mientras más anticipada, más cancela), required_car_parking_spaces porque el que pide estacionamiento casi nunca cancela, deposito_no_reemb porque aunque suene raro los depósitos no reembolsables se cancelan más, y booking_changes porque el que le hace cambios a su reserva cancela menos. Y total_of_special_requests sirve para las dos cosas, el que pide cosas especiales cancela menos y paga un poco más.

Algunas que dejamos fuera a propósito fueron arrival_date_year, que tiene correlación con adr pero solo porque los precios subieron entre 2015 y 2017 y en otro año no serviría, arrival_date_week_number que es parecida a temporada_alta pero en número y no va en línea recta con el precio, stays_in_week_nights y stays_in_weekend_nights que tienen correlación muy baja con las dos, y assigned_room_type, porque la habitación que te asignan se sabe recién al llegar al hotel y para predecir cancelación sería medio trampa.

In [14]:
X_cols = ["lead_time", "adults", "children", "total_of_special_requests",
          "required_car_parking_spaces", "hotel_city", "temporada_alta",
          "deposito_no_reemb", "is_repeated_guest", "booking_changes"]

X_train = train[X_cols]
X_val = val[X_cols]
X_test = test[X_cols]

y1_train = train["adr"]
y1_val = val["adr"]
y1_test = test["adr"]

y2_train = train["is_canceled"]
y2_val = val["is_canceled"]
y2_test = test["is_canceled"]

print(X_train.describe().round(2))

       lead_time    adults  children  total_of_special_requests  \
count   59909.00  59909.00  59909.00                   59909.00   
mean       80.83      1.88      0.14                       0.70   
std        86.06      0.50      0.46                       0.83   
min         0.00      0.00      0.00                       0.00   
25%        12.00      2.00      0.00                       0.00   
50%        51.00      2.00      0.00                       0.00   
75%       126.00      2.00      0.00                       1.00   
max       709.00      4.00      3.00                       5.00   

       required_car_parking_spaces  hotel_city  temporada_alta  \
count                     59909.00    59909.00        59909.00   
mean                          0.08        0.61            0.33   
std                           0.28        0.49            0.47   
min                           0.00        0.00            0.00   
25%                           0.00        0.00            0.00   


### 4.4 Escalado
Las variables tienen escalas muy distintas, lead_time llega a más de 600 y las binarias son 0 o 1, así que las escalamos para que la logística funcione mejor y para poder comparar los coeficientes entre ellos. El StandardScaler lo ajustamos solo con train y después lo usamos igual en val y test.

In [15]:
scaler = StandardScaler()
scaler.fit(X_train)

X_train_esc = pd.DataFrame(scaler.transform(X_train), columns=X_cols)
X_val_esc = pd.DataFrame(scaler.transform(X_val), columns=X_cols)
X_test_esc = pd.DataFrame(scaler.transform(X_test), columns=X_cols)

## 5. Selección de variables (solo con entrenamiento): top 5 y top 3
Lo que hicimos aquí fue ver la correlación de cada una de las 10 variables con su Y, pero en valor absoluto porque nos da lo mismo si la relación es positiva o negativa, lo que importa es qué tan fuerte es. Esto lo hicimos solo con train y por separado para cada modelo, porque una variable puede servir harto para el precio y casi nada para saber si cancelan, o al revés.

Usamos la correlación porque es fácil de sacar y como los dos modelos son lineales tiene sentido quedarnos con las variables que tienen una relación más en línea recta con lo que queremos predecir, así que de cada lista tomamos las 5 primeras y las 3 primeras.

In [16]:
corr_adr = X_train.corrwith(y1_train).abs().sort_values(ascending=False)
print(corr_adr.round(3))

temporada_alta                 0.436
children                       0.362
adults                         0.328
total_of_special_requests      0.150
is_repeated_guest              0.127
hotel_city                     0.122
deposito_no_reemb              0.045
required_car_parking_spaces    0.039
booking_changes                0.021
lead_time                      0.008
dtype: float64


In [17]:
corr_cancel = X_train.corrwith(y2_train).abs().sort_values(ascending=False)
print(corr_cancel.round(3))

required_car_parking_spaces    0.187
lead_time                      0.180
deposito_no_reemb              0.164
total_of_special_requests      0.122
booking_changes                0.095
adults                         0.083
is_repeated_guest              0.081
hotel_city                     0.075
temporada_alta                 0.064
children                       0.063
dtype: float64


In [18]:
top5_adr = list(corr_adr.index[:5])
top3_adr = list(corr_adr.index[:3])
top5_cancel = list(corr_cancel.index[:5])
top3_cancel = list(corr_cancel.index[:3])

print("adr top 5:", top5_adr)
print("adr top 3:", top3_adr)
print("is_canceled top 5:", top5_cancel)
print("is_canceled top 3:", top3_cancel)

adr top 5: ['temporada_alta', 'children', 'adults', 'total_of_special_requests', 'is_repeated_guest']
adr top 3: ['temporada_alta', 'children', 'adults']
is_canceled top 5: ['required_car_parking_spaces', 'lead_time', 'deposito_no_reemb', 'total_of_special_requests', 'booking_changes']
is_canceled top 3: ['required_car_parking_spaces', 'lead_time', 'deposito_no_reemb']


Lo que salió fue bastante distinto para cada Y. Para el adr las 3 mejores son temporada_alta, children y adults y en el top 5 se suman total_of_special_requests e is_repeated_guest, lo que tiene sentido porque en verano sube el precio y si viene más gente la habitación es más grande y más cara. En cambio para is_canceled las 3 mejores son required_car_parking_spaces, lead_time y deposito_no_reemb y en el top 5 se suman total_of_special_requests y booking_changes, y ahí temporada_alta y children, que eran las mejores para el precio, quedan al final. Lo mismo pasa al revés con lead_time, que con el adr da 0.008, o sea prácticamente nada.

La única que está en el top 5 de los dos es total_of_special_requests, que era justo por lo que la habíamos elegido entre las 10. También nos dimos cuenta de que las correlaciones con is_canceled son todas bajas (la más alta es 0.187) y con adr la mejor llega a 0.436, entonces ya pensamos que con pocas variables a la logística le iba a costar más que a la lineal. El tema de la correlación es que mira cada variable sola, entonces si dos dicen casi lo mismo, como adults y children, igual pueden quedar las dos arriba.

## 6. Tarea A: regresión lineal para Y1 (10, 5 y 3 variables)
Aquí entrenamos 3 regresiones lineales para el adr, una con las 10 variables, otra con el top 5 y otra con el top 3, todas solo con train.

In [19]:
lin_10 = LinearRegression()
lin_10.fit(X_train_esc[X_cols], y1_train)

lin_5 = LinearRegression()
lin_5.fit(X_train_esc[top5_adr], y1_train)

lin_3 = LinearRegression()
lin_3.fit(X_train_esc[top3_adr], y1_train)

print(pd.Series(lin_10.coef_, index=X_cols).round(2))

lead_time                      -6.45
adults                         12.66
children                       15.60
total_of_special_requests       3.81
required_car_parking_spaces     2.12
hotel_city                      6.68
temporada_alta                 19.43
deposito_no_reemb               0.33
is_repeated_guest              -2.61
booking_changes                 1.62
dtype: float64


Mirando los coeficientes del de 10, lo que más sube el precio es temporada_alta y después children y adults, que son justo las 3 del top 3, así que la correlación y el modelo dicen lo mismo. lead_time sale negativo, o sea que reservar con mucha anticipación sale un poco más barato, y deposito_no_reemb casi no aporta nada (0.33).

### 6.1 Comparación entrenamiento vs validación

In [20]:
modelos = [lin_10, lin_5, lin_3]
columnas = [X_cols, top5_adr, top3_adr]
nombres = ["10 variables", "5 variables", "3 variables"]

for i in range(3):
    pred_train = modelos[i].predict(X_train_esc[columnas[i]])
    pred_val = modelos[i].predict(X_val_esc[columnas[i]])
    print(nombres[i])
    print("  R2   train:", round(r2_score(y1_train, pred_train), 3), " val:", round(r2_score(y1_val, pred_val), 3))
    print("  RMSE train:", round(mean_squared_error(y1_train, pred_train) ** 0.5, 2), " val:", round(mean_squared_error(y1_val, pred_val) ** 0.5, 2))
    print("  MAE  train:", round(mean_absolute_error(y1_train, pred_train), 2), " val:", round(mean_absolute_error(y1_val, pred_val), 2))

10 variables
  R2   train: 0.398  val: 0.398
  RMSE train: 38.98  val: 39.05
  MAE  train: 29.58  val: 29.6
5 variables
  R2   train: 0.364  val: 0.363
  RMSE train: 40.07  val: 40.15
  MAE  train: 30.87  val: 30.88
3 variables
  R2   train: 0.357  val: 0.356
  RMSE train: 40.31  val: 40.38
  MAE  train: 31.09  val: 31.12


Usamos R2 para ver cuánto del precio logra explicar el modelo, y RMSE y MAE para ver cuánto se equivoca en euros. El MAE es el error promedio y el RMSE castiga más los errores grandes, por eso siempre da un poco más alto.

Lo que vimos aquí fue que el de 10 variables es el mejor, con un R2 de 0.40 en validación y un error promedio de unos 30 euros por noche, y al sacar variables empeora de a poco, con 5 baja a 0.36 y con 3 queda en 0.36 también, con un error de 31 euros. La diferencia entre 5 y 3 es muy chica, así que total_of_special_requests e is_repeated_guest casi no le aportan, y lo que más se pierde es al pasar de 10 a 5, o sea que hotel_city y lead_time, que no estaban en el top 5, igual ayudaban un poco junto con las otras.

Lo otro es que en los tres casos train y validación dan prácticamente lo mismo, la diferencia está en el tercer decimal, entonces el modelo no se está aprendiendo los datos de memoria y no hay overfitting. Lo que sí pasa es que ni con las 10 explica más del 40%, así que se queda corto, más que nada porque le faltan cosas que influyen harto en el precio, como el tipo de habitación.

### 6.2 Evaluación en prueba de las 2 mejores versiones

In [21]:
pred_10 = lin_10.predict(X_test_esc[X_cols])
pred_5 = lin_5.predict(X_test_esc[top5_adr])

print("10 variables  R2:", round(r2_score(y1_test, pred_10), 3), " RMSE:", round(mean_squared_error(y1_test, pred_10) ** 0.5, 2), " MAE:", round(mean_absolute_error(y1_test, pred_10), 2))
print("5 variables   R2:", round(r2_score(y1_test, pred_5), 3), " RMSE:", round(mean_squared_error(y1_test, pred_5) ** 0.5, 2), " MAE:", round(mean_absolute_error(y1_test, pred_5), 2))

10 variables  R2: 0.37  RMSE: 39.62  MAE: 30.01
5 variables   R2: 0.341  RMSE: 40.53  MAE: 31.11


Las dos mejores en validación fueron la de 10 y la de 5, así que esas son las que probamos con test. La de 10 sigue siendo la mejor con un R2 de 0.37 y un error promedio de 30 euros, y la de 5 queda con 0.34 y 31 euros, o sea que el orden es el mismo que en validación.

En test las dos bajan un poco el R2 comparado con validación (de 0.40 a 0.37 en la de 10), pero el error en euros casi no cambia, así que el modelo se porta parecido con datos que nunca vio.

## 7. Tarea B: regresión logística para Y2 (10, 5 y 3 variables)
Aquí hicimos lo mismo pero ahora para is_canceled, entrenamos 3 regresiones logísticas, una con las 10, otra con el top 5 y otra con el top 3 de la cancelación, todas solo con train. Le pusimos max_iter=1000 porque con el valor que trae por defecto a veces no alcanza a terminar.

In [22]:
log_10 = LogisticRegression(max_iter=1000)
log_10.fit(X_train_esc[X_cols], y2_train)

log_5 = LogisticRegression(max_iter=1000)
log_5.fit(X_train_esc[top5_cancel], y2_train)

log_3 = LogisticRegression(max_iter=1000)
log_3.fit(X_train_esc[top3_cancel], y2_train)

print(pd.Series(log_10.coef_[0], index=X_cols).round(2))

lead_time                      0.32
adults                         0.17
children                       0.18
total_of_special_requests     -0.32
required_car_parking_spaces   -2.20
hotel_city                     0.09
temporada_alta                 0.06
deposito_no_reemb              0.35
is_repeated_guest             -0.15
booking_changes               -0.30
dtype: float64


En los coeficientes del de 10 lo que más baja la probabilidad de cancelar es required_car_parking_spaces por lejos (-2.2), o sea que el que pide estacionamiento casi nunca cancela, y lo que más la sube es deposito_no_reemb y lead_time. Lo del depósito nos llamó la atención porque uno pensaría que si pagaste y no te devuelven la plata no cancelas, pero en estos datos pasa lo contrario, y creemos que es porque son reservas de grupos o agencias que se caen todas juntas.

### 7.1 Comparación entrenamiento vs validación

In [23]:
modelos = [log_10, log_5, log_3]
columnas = [X_cols, top5_cancel, top3_cancel]

for i in range(3):
    pred_train = modelos[i].predict(X_train_esc[columnas[i]])
    pred_val = modelos[i].predict(X_val_esc[columnas[i]])
    prob_train = modelos[i].predict_proba(X_train_esc[columnas[i]])[:, 1]
    prob_val = modelos[i].predict_proba(X_val_esc[columnas[i]])[:, 1]
    print(nombres[i])
    print("  accuracy  train:", round(accuracy_score(y2_train, pred_train), 3), " val:", round(accuracy_score(y2_val, pred_val), 3))
    print("  precision train:", round(precision_score(y2_train, pred_train), 3), " val:", round(precision_score(y2_val, pred_val), 3))
    print("  recall    train:", round(recall_score(y2_train, pred_train), 3), " val:", round(recall_score(y2_val, pred_val), 3))
    print("  F1        train:", round(f1_score(y2_train, pred_train), 3), " val:", round(f1_score(y2_val, pred_val), 3))
    print("  AUC       train:", round(roc_auc_score(y2_train, prob_train), 3), " val:", round(roc_auc_score(y2_val, prob_val), 3))

10 variables
  accuracy  train: 0.74  val: 0.746
  precision train: 0.642  val: 0.663
  recall    train: 0.149  val: 0.155


  F1        train: 0.242  val: 0.251


  AUC       train: 0.714  val: 0.717
5 variables
  accuracy  train: 0.73  val: 0.734
  precision train: 0.59  val: 0.604
  recall    train: 0.1  val: 0.1
  F1        train: 0.171  val: 0.172
  AUC       train: 0.706  val: 0.709
3 variables
  accuracy  train: 0.728  val: 0.732
  precision train: 0.616  val: 0.655
  recall    train: 0.062  val: 0.059


  F1        train: 0.112  val: 0.108


  AUC       train: 0.682  val: 0.687


In [24]:
print(confusion_matrix(y2_val, log_10.predict(X_val_esc[X_cols])))

[[9023  278]
 [2989  548]]


Como las clases no están parejas no nos podíamos quedar solo con el accuracy, porque un modelo que diga que nadie cancela ya tendría 0.72. Por eso miramos también la precision, que es de las que dijo que cancelan cuántas cancelaron de verdad, el recall, que es de las que cancelaron cuántas pilló, el F1 que junta las dos, y el AUC, que dice qué tan bien ordena las reservas de más a menos probable de cancelar sin importar el corte.

Lo que vimos aquí fue que el de 10 variables es el mejor en todo, con un accuracy de 0.75 en validación, que es apenas un poco más que decir que nadie cancela. El problema está en el recall, que es 0.15, o sea que de todas las reservas que se cancelan solo pilla como el 15%, y en la matriz de confusión (filas son lo real y columnas lo que predijo) se ve clarito, de las 3537 cancelaciones reales solo le achunta a 548 y las otras 2989 las dice como que no cancelan. Esto pasa porque hay muchas más reservas que no se cancelan y el modelo se va para ese lado. Al sacar variables se pone peor, con 5 el recall baja a 0.10 y con 3 a 0.06, y el AUC baja de 0.72 a 0.71 y a 0.69.

En los tres casos train y validación dan casi lo mismo, así que tampoco hay overfitting acá. Pasa lo contrario, el modelo se queda corto porque con estas variables no le alcanza para separar bien las que cancelan de las que no, que es lo que ya veíamos venir por las correlaciones tan bajas de la sección 5.

### 7.2 Evaluación en prueba de las 2 mejores versiones

In [25]:
pred_10 = log_10.predict(X_test_esc[X_cols])
prob_10 = log_10.predict_proba(X_test_esc[X_cols])[:, 1]
pred_5 = log_5.predict(X_test_esc[top5_cancel])
prob_5 = log_5.predict_proba(X_test_esc[top5_cancel])[:, 1]

print("10 variables")
print("  accuracy:", round(accuracy_score(y2_test, pred_10), 3), " precision:", round(precision_score(y2_test, pred_10), 3), " recall:", round(recall_score(y2_test, pred_10), 3), " F1:", round(f1_score(y2_test, pred_10), 3), " AUC:", round(roc_auc_score(y2_test, prob_10), 3))
print("5 variables")
print("  accuracy:", round(accuracy_score(y2_test, pred_5), 3), " precision:", round(precision_score(y2_test, pred_5), 3), " recall:", round(recall_score(y2_test, pred_5), 3), " F1:", round(f1_score(y2_test, pred_5), 3), " AUC:", round(roc_auc_score(y2_test, prob_5), 3))

10 variables
  accuracy: 0.741  precision: 0.663  recall: 0.15  F1: 0.244  AUC: 0.712
5 variables
  accuracy: 0.73  precision: 0.611  recall: 0.099  F1: 0.17  AUC: 0.7


Las dos mejores en validación fueron la de 10 y la de 5, así que esas probamos con test. La de 10 queda con accuracy 0.74, recall 0.15, F1 0.24 y AUC 0.71, y la de 5 con accuracy 0.73, recall 0.10, F1 0.17 y AUC 0.70, o sea casi lo mismo que en validación y en el mismo orden.

Entonces el modelo se porta igual con datos que no vio, lo que confirma que no hay overfitting, pero también que sigue con el mismo problema de pillar pocas cancelaciones. Si quisiéramos que pille más se podría bajar el corte de 0.5 o balancear las clases, pero lo dejamos así para comparar las tres versiones de la misma forma.

## 8. Análisis: cantidad de variables, generalización, overfitting/underfitting

Lo que vimos en general es que mientras menos variables le pasamos al modelo peor le va, pero no le pega igual a los dos. En la lineal casi no se nota, con las 10 el R2 da 0.40 y con 3 da 0.36, o sea que con temporada_alta, children y adults ya hace casi todo el trabajo y las otras 7 ayudan poquito. En la logística sí se nota harto, el recall baja de 0.15 a 0.06 y el F1 de 0.25 a 0.11, y eso es porque ninguna variable sola dice mucho sobre si alguien va a cancelar, entonces el modelo necesita varias juntas para poder adivinar algo.

Después en todos los casos train y validación dan casi igual, la diferencia es de milésimas y a veces validación hasta da un poco mejor, y en test pasa lo mismo. Así que ninguno de los 6 modelos tiene overfitting, que era lo esperable porque son modelos simples, con máximo 10 variables y con casi 60 mil filas para entrenar, entonces no tienen cómo aprenderse los datos de memoria.

Lo que sí hay es underfitting, más que nada en la logística y en los modelos con menos variables. Los modelos se quedan cortos, la lineal no pasa del 40% y la logística pilla muy pocas cancelaciones, y como les va igual de mal en train que en validación el problema no es que fallen con datos nuevos sino que no aprenden lo suficiente ni con los de entrenamiento. Y si les sacamos variables es peor porque les damos menos con qué trabajar.

Al final nos quedamos con los de 10 variables en los dos casos, porque son los que mejor salieron en validación y en test y como no hay overfitting no tiene sentido usar menos. Para que salga mejor habría que darle más datos en vez de quitarle, como el tipo de habitación, el segmento o el país, o probar otro tipo de modelo que no sea una línea recta, porque por ejemplo el precio sube en verano y baja en invierno y eso a la lineal le cuesta.